# 03 - 多 Agent 组合研判策略研究与回测

策略思路：由多个 Agent 打分后集成决策（趋势/动量/均值回归/波动突破），再执行买卖与回测。

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
for p in [cwd / 'src', cwd.parent / 'src']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from quant_research.db import query_df

In [ ]:
symbol = 'US:NVDA'
start_date = '2023-01-01'
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
fee_rate = 0.0003
slippage = 0.001

agent_weights = {
    'trend_agent': 1.0,
    'momentum_agent': 1.0,
    'mean_reversion_agent': 0.8,
    'breakout_agent': 1.2,
}
buy_score_th = 1.5
sell_score_th = -0.5

In [ ]:
df = query_df(f"""
select trade_date, symbol, close, high, low, volume
from daily_bars
where symbol = '{symbol}'
  and trade_date between toDate('{start_date}') and toDate('{end_date}')
order by trade_date
""")
df['trade_date'] = pd.to_datetime(df['trade_date'])
df.head()

In [ ]:
def add_agents(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    # Agent 1: 趋势（MA10 vs MA30）
    out['ma10'] = out['close'].rolling(10).mean()
    out['ma30'] = out['close'].rolling(30).mean()
    out['trend_agent'] = np.where(out['ma10'] > out['ma30'], 1, -1)

    # Agent 2: 动量（20日收益）
    out['mom20'] = out['close'].pct_change(20)
    out['momentum_agent'] = np.where(out['mom20'] > 0, 1, -1)

    # Agent 3: 均值回归（RSI）
    delta = out['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-12)
    out['rsi'] = 100 - 100 / (1 + rs)
    out['mean_reversion_agent'] = np.where(out['rsi'] < 30, 1, np.where(out['rsi'] > 70, -1, 0))

    # Agent 4: 波动突破（20日新高）
    out['high20'] = out['high'].rolling(20).max()
    out['low20'] = out['low'].rolling(20).min()
    out['breakout_agent'] = np.where(out['close'] > out['high20'].shift(1), 1, np.where(out['close'] < out['low20'].shift(1), -1, 0))

    return out


def backtest_multi_agent(data: pd.DataFrame):
    out = add_agents(data)
    out['ret'] = out['close'].pct_change().fillna(0.0)

    score = (
        agent_weights['trend_agent'] * out['trend_agent'] +
        agent_weights['momentum_agent'] * out['momentum_agent'] +
        agent_weights['mean_reversion_agent'] * out['mean_reversion_agent'] +
        agent_weights['breakout_agent'] * out['breakout_agent']
    )
    out['agent_score'] = score

    pos = np.zeros(len(out), dtype=int)
    for i in range(1, len(out)):
        pos[i] = pos[i - 1]
        if out['agent_score'].iat[i] >= buy_score_th:
            pos[i] = 1
        elif out['agent_score'].iat[i] <= sell_score_th:
            pos[i] = 0

    out['position'] = pd.Series(pos, index=out.index).shift(1).fillna(0).astype(int)
    out['trade_flag'] = out['position'].diff().abs().fillna(0)
    cost = out['trade_flag'] * (fee_rate + slippage)

    out['strategy_ret'] = out['position'] * out['ret'] - cost
    out['strategy_curve'] = (1 + out['strategy_ret']).cumprod()
    out['benchmark_curve'] = (1 + out['ret']).cumprod()

    days = max(len(out), 1)
    ann_factor = 252 / days
    total_return = out['strategy_curve'].iat[-1] - 1
    cagr = (1 + total_return) ** ann_factor - 1
    mdd = (out['strategy_curve'] / out['strategy_curve'].cummax() - 1).min()
    sharpe = np.sqrt(252) * out['strategy_ret'].mean() / (out['strategy_ret'].std() + 1e-12)

    metrics = {
        'total_return': float(total_return),
        'cagr': float(cagr),
        'max_drawdown': float(mdd),
        'sharpe': float(sharpe),
        'num_trades': int(out['trade_flag'].sum()),
    }
    return out, metrics

In [ ]:
bt, metrics = backtest_multi_agent(df)
pd.DataFrame([{'symbol': symbol, **metrics}])

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(bt['trade_date'], bt['close'], label='Close')
axes[0].set_title(f'Multi-Agent Strategy - {symbol}')
axes[0].legend()

axes[1].plot(bt['trade_date'], bt['agent_score'], label='Agent Score')
axes[1].axhline(buy_score_th, color='red', linestyle='--', label='Buy Threshold')
axes[1].axhline(sell_score_th, color='black', linestyle='--', label='Sell Threshold')
axes[1].legend()

axes[2].plot(bt['trade_date'], bt['strategy_curve'] - 1, label='Strategy CumReturn')
axes[2].plot(bt['trade_date'], bt['benchmark_curve'] - 1, label='Benchmark CumReturn')
axes[2].legend()

plt.xticks(rotation=30)
plt.tight_layout()